In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error



In [ ]:
DATA = Path("../data")
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)



In [ ]:
# ======================
# 1. ЗАГРУЗКА РЯДА
# ======================
df = pd.read_csv(DATA / "store_sales.csv", parse_dates=["date"])
df = df.sort_values(["store_id", "date"]).reset_index(drop=True)



In [ ]:
print("rows:", df.shape[0])
print("stores:", sorted(df["store_id"].unique()))
df.head()



In [ ]:
# Берём магазин 1
g = df[df["store_id"] == 1].copy()

plt.figure(figsize=(11, 3.5))
plt.plot(g["date"], g["sales"], color="#4c72b0")
plt.title("Store 1 — raw sales series", fontsize=14, pad=10)
plt.xlabel("Date")
plt.ylabel("Sales")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# ======================
# 2. LAG FEATURES
# ======================
g["lag7"] = g["sales"].shift(7)
g["lag14"] = g["sales"].shift(14)
g = g.dropna().reset_index(drop=True)
print("rows after lags:", len(g))
g.head()



In [ ]:
# Связь лагов с целевой
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].scatter(g["lag7"], g["sales"], alpha=0.5, c="#4c72b0")
axes[0].set_title("sales vs lag7")
axes[0].set_xlabel("lag7")
axes[0].set_ylabel("sales")
axes[1].scatter(g["lag14"], g["sales"], alpha=0.5, c="#55a868")
axes[1].set_title("sales vs lag14")
axes[1].set_xlabel("lag14")
plt.tight_layout()
plt.show()



In [ ]:
# ======================
# 3. TRAIN / TEST SPLIT
# ======================
split = int(len(g) * 0.8)
train, test = g.iloc[:split], g.iloc[split:]
feats = ["lag7", "lag14"]
print("train:", len(train), "test:", len(test))



In [ ]:
# ======================
# 4. RANDOM FOREST FORECAST
# ======================
model = RandomForestRegressor(n_estimators=100, random_state=0)
model.fit(train[feats], train["sales"])
pred = model.predict(test[feats])
mae = mean_absolute_error(test["sales"], pred)
print("MAE:", round(mae, 2))



In [ ]:
# Основной forecast plot
plt.figure(figsize=(10, 3.5))
plt.plot(test["date"], test["sales"], label="actual", color="blue")
plt.plot(test["date"], pred, label="pred", color="red", alpha=0.85)
plt.title("Store 1 lag-feature forecast", fontsize=14, pad=10)
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / "forecast_store1.png", dpi=120, bbox_inches="tight")
plt.show()



In [ ]:
# Ошибки во времени
err = np.abs(test["sales"].values - pred)

plt.figure(figsize=(10, 3))
plt.plot(test["date"], err, color="#c44e52")
plt.title("Absolute error over time", fontsize=13, pad=8)
plt.ylabel("|error|")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# Важность лагов
imp = pd.Series(model.feature_importances_, index=feats)

plt.figure(figsize=(5, 3.5))
sns.barplot(x=imp.index, y=imp.values, palette="viridis")
plt.title("RF feature importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.show()



In [ ]:
# Residual histogram
resid = test["sales"].values - pred

plt.figure(figsize=(7, 4))
sns.histplot(resid, bins=25, color="#4c72b0", edgecolor="black")
plt.axvline(0, color="red", linestyle="--")
plt.title("Residual distribution", fontsize=13, pad=8)
plt.xlabel("actual - pred")
plt.tight_layout()
plt.show()



In [ ]:
# Сводка
pd.Series({
    "MAE": round(mae, 2),
    "mean_actual": round(float(test["sales"].mean()), 2),
    "mean_pred": round(float(pred.mean()), 2),
})



In [ ]:
# Сравнение MAE с наивным прогнозом (lag7 как baseline)
naive = test["lag7"].values
mae_naive = mean_absolute_error(test["sales"], naive)
print("RF MAE:", round(mae, 2))
print("Naive lag7 MAE:", round(mae_naive, 2))
print("Lift vs naive:", round(mae_naive - mae, 2))
